# sigma-controller demo

PID controller for sigma — the advertiser's reach parameter.

The advertiser sets a margin. The exchange observes conversion rates at different distances. The controller adjusts sigma to track the boundary where conversions remain profitable.

Run each cell to see it work.

In [ ]:
from pid import SigmaController, DistanceBin, validate_histogram
import math
import random
from unittest.mock import patch

## 1. Basic usage

Create a controller and feed it a distance histogram.

In [ ]:
controller = SigmaController(target_rate=0.05, sigma=2.0)

# A histogram from the publisher: conversion counts per distance bin
histogram = [
    DistanceBin(distance=0.25, impressions=200, conversions=180),
    DistanceBin(distance=0.75, impressions=200, conversions=150),
    DistanceBin(distance=1.25, impressions=200, conversions=80),
    DistanceBin(distance=1.75, impressions=200, conversions=30),
    DistanceBin(distance=2.25, impressions=200, conversions=8),
    DistanceBin(distance=2.75, impressions=200, conversions=2),
]

print("Conversion rates by distance:")
for b in histogram:
    print(f"  d={b.distance:.2f}: {b.conversion_rate:.0%} ({b.conversions}/{b.impressions})")

print(f"\nInitial sigma: {controller.sigma}")
print(f"Target boundary rate: {controller.target_rate:.0%}")

In [ ]:
# One update step
with patch("pid.time") as mock_time:
    mock_time.monotonic.return_value = 1.0
    controller._prev_time = 0.0
    new_sigma = controller.update(histogram)

print(f"Sigma after one update: {new_sigma:.4f}")

## 2. Bin suppression

Bins with fewer than 11 impressions are automatically suppressed (CMS cell suppression policy).

In [ ]:
sparse_histogram = [
    DistanceBin(distance=0.5, impressions=200, conversions=180),
    DistanceBin(distance=1.0, impressions=50, conversions=25),
    DistanceBin(distance=1.5, impressions=8, conversions=3),   # suppressed (< 11)
    DistanceBin(distance=2.0, impressions=5, conversions=1),   # suppressed (< 11)
    DistanceBin(distance=2.5, impressions=200, conversions=4),
]

safe = validate_histogram(sparse_histogram)
print(f"Bins before suppression: {len(sparse_histogram)}")
print(f"Bins after suppression: {len(safe)}")
print(f"\nSurviving bins:")
for b in safe:
    print(f"  d={b.distance:.1f}: {b.impressions} impressions")

## 3. Convergence

Start sigma too high (4.0) and too low (0.5) for the same underlying Gaussian. Both converge to the same equilibrium.

In [ ]:
import matplotlib.pyplot as plt

def make_histogram(sigma_true, n_bins=40, max_dist=5.0, impressions=200, noise=0.02):
    bins = []
    for i in range(n_bins):
        d = (i + 0.5) * max_dist / n_bins
        rate = math.exp(-d**2 / (2 * sigma_true**2))
        rate = max(0, rate + random.gauss(0, noise))
        conv = max(0, min(impressions, int(rate * impressions)))
        bins.append(DistanceBin(distance=d, impressions=impressions, conversions=conv))
    return bins

true_sigma = 1.5
target = 0.10
n_steps = 300

results = {}
for start in [0.5, 4.0]:
    random.seed(42)
    ctrl = SigmaController(target_rate=target, sigma=start, kp=0.3, ki=0.02, kd=0.08)
    history = [ctrl.sigma]

    with patch("pid.time") as mock_time:
        t = 0.0
        ctrl._prev_time = t
        for _ in range(n_steps):
            t += 1.0
            mock_time.monotonic.return_value = t
            h = make_histogram(true_sigma)
            ctrl.update(h)
            history.append(ctrl.sigma)

    results[start] = history

# Theoretical equilibrium
eq = true_sigma * math.sqrt(-2 * math.log(target))

plt.figure(figsize=(10, 5))
for start, history in results.items():
    plt.plot(history, label=f"start={start}", alpha=0.8)
plt.axhline(y=eq, color="gray", linestyle=":", alpha=0.5, label=f"Equilibrium={eq:.2f}")
plt.xlabel("Update Steps")
plt.ylabel("σ")
plt.title(f"Sigma convergence (true σ={true_sigma}, target={target:.0%})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

for start, history in results.items():
    print(f"Start={start} → final σ={history[-1]:.3f} (equilibrium={eq:.3f})")

## 4. Try your own parameters

Change `my_true_sigma` and `my_target` and re-run.

In [ ]:
my_true_sigma = 2.0   # <-- change this (0.5 to 5.0)
my_target = 0.05       # <-- change this (0.01 to 0.50)
my_start = 5.0         # <-- change this (starting sigma guess)

random.seed(42)
ctrl = SigmaController(target_rate=my_target, sigma=my_start, kp=0.3, ki=0.02, kd=0.08)
history = [ctrl.sigma]
boundary_rates = []

with patch("pid.time") as mock_time:
    t = 0.0
    ctrl._prev_time = t
    for _ in range(400):
        t += 1.0
        mock_time.monotonic.return_value = t
        h = make_histogram(my_true_sigma)
        boundary_rates.append(ctrl._boundary_conversion_rate(validate_histogram(h)))
        ctrl.update(h)
        history.append(ctrl.sigma)

eq = my_true_sigma * math.sqrt(-2 * math.log(my_target)) if my_target > 0 else float('inf')

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(history, color="steelblue")
axes[0].axhline(y=eq, color="gray", linestyle=":", alpha=0.5, label=f"Equilibrium={eq:.2f}")
axes[0].set_ylabel("σ")
axes[0].set_title(f"Sigma convergence (true σ={my_true_sigma}, target={my_target:.0%})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(boundary_rates, color="steelblue")
axes[1].axhline(y=my_target, color="red", linestyle="--", alpha=0.5, label="Target")
axes[1].set_ylabel("Boundary Conversion Rate")
axes[1].set_xlabel("Update Steps")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final σ: {history[-1]:.3f}")
print(f"Equilibrium: {eq:.3f}")
print(f"Error: {abs(history[-1] - eq):.3f}")

## 5. Competitor entry

A competitor enters at step 150. The effective conversion curve narrows by 40%. Watch sigma contract and re-stabilize.

In [ ]:
random.seed(42)
ctrl = SigmaController(target_rate=0.10, sigma=2.0, kp=0.3, ki=0.02, kd=0.08)
history = [ctrl.sigma]
shock_at = 150

with patch("pid.time") as mock_time:
    t = 0.0
    ctrl._prev_time = t
    for i in range(300):
        t += 1.0
        mock_time.monotonic.return_value = t
        effective = 1.5 * 0.6 if i >= shock_at else 1.5
        h = make_histogram(effective)
        ctrl.update(h)
        history.append(ctrl.sigma)

plt.figure(figsize=(10, 4))
plt.plot(history, color="steelblue")
plt.axvline(x=shock_at, color="red", linestyle="--", alpha=0.5, label="Competitor enters")
plt.xlabel("Update Steps")
plt.ylabel("σ")
plt.title("Sigma recovery after competitor entry")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Initial estimate

`estimate_sigma_from_curve` bootstraps a reasonable starting sigma from a single histogram, before the PID loop takes over.

In [ ]:
random.seed(42)
ctrl = SigmaController(target_rate=0.10, sigma=5.0)  # bad initial guess

h = make_histogram(1.5, impressions=1000, noise=0.0)
estimated = ctrl.estimate_sigma_from_curve(h)

print(f"True σ: 1.5")
print(f"Bad initial guess: 5.0")
print(f"Estimated from curve: {estimated:.3f}")
print(f"\nClose enough for the PID to converge quickly.")